# RSNA Knee Abnormality Detection - Training (Simplified MedDINOv3)

Training with frozen MedDINOv3 backbone. Images auto-padded to 14x multiples.

In [10]:
# !pip install pydicom timm albumentations scikit-learn tqdm

import os, math, random, warnings
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import timm
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import multiprocessing as mp
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

from model_meddinov3_simplified import KneeAbnormalityModelWithMedDINOv3, pad_image

warnings.filterwarnings('ignore')
print('✓ All imports successful')

✓ All imports successful


In [11]:
class Config:
    KAGGLE_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection')
    PYTHON_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection')
    TRAIN_IMG_DIR   = KAGGLE_DIR / 'train_series'
    LABELS_CSV      = KAGGLE_DIR / 'train_series.csv'
    CHECKPOINT_DIR  = PYTHON_DIR

    # Image processing (padded to 14x multiples for MedDINOv3)
    PATCH_SIZE_DINO = 14  # MedDINOv3 uses 14x14 patches internally
    MAX_SLICES      = 12
    LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
              'Medial OA','Lateral OA','PF OA','Effusion',
              'Synovitis',"Baker's",'Contusion','Fracture']
    NUM_LABELS   = 12
    PLANE_TO_IDX = {'Sagittal': 0, 'Axial': 1, 'Coronal': 2}
    NUM_PLANES   = 3

    # Model dimensions
    SLICE_DIM    = 512
    META_DIM     = 256
    SLICE_ENC_HEADS  = 8
    SLICE_ENC_LAYERS = 2
    DROPOUT      = 0.3

    # Training hyperparameters
    NUM_EPOCHS   = 30
    BATCH_SIZE   = 2
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    WARMUP_EPOCHS = 3
    NUM_WORKERS  = 4
    VAL_SPLIT    = 0.2
    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_AMP      = True

cfg = Config()
print(f'Device: {cfg.DEVICE}')
print(f'Batch size: {cfg.BATCH_SIZE}')
print(f'MedDINOv3 frozen, training decoder only')

Device: cuda
Batch size: 2
MedDINOv3 frozen, training decoder only


In [12]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

In [13]:
# ── Training / Validation Dataset ─────────────────────────────────────────────
class KneeTrainDataset(Dataset):
    def __init__(self, df_labels: pd.DataFrame, img_dir: Path, cfg, split='train'):
        self.cfg = cfg
        self.img_dir = img_dir
        
        # Group by StudyInstanceUID
        study_ids = df_labels['StudyInstanceUID'].unique()
        n = len(study_ids)
        split_idx = int(n * (1 - cfg.VAL_SPLIT))
        
        if split == 'train':
            study_ids = study_ids[:split_idx]
        else:
            study_ids = study_ids[split_idx:]
        
        self.studies = []
        for study_id in study_ids:
            study_df = df_labels[df_labels['StudyInstanceUID'] == study_id]
            series_list = []
            for _, row in study_df.iterrows():
                series_list.append({
                    'series_id': row['SeriesInstanceUID'],
                    'plane': row['Anatomical_Plane'],
                    'plane_id': cfg.PLANE_TO_IDX.get(row['Anatomical_Plane'], 0),
                    'fluid_sensitive': int(row.get('fluid_sensitive', 0)),
                    'fat_suppression': int(row.get('fat_suppression', 0))
                })
            self.studies.append({'study_id': study_id, 'series': series_list})
        
        # Load labels
        self.labels_df = df_labels.set_index('StudyInstanceUID')

    def _load_series(self, series_id: str) -> torch.Tensor:
        series_dir = self.img_dir / series_id
        dcm_files = sorted(series_dir.glob('*.dcm'))
        
        images = []
        for dcm_file in dcm_files[:self.cfg.MAX_SLICES]:
            dcm = pydicom.dcmread(dcm_file)
            img = dcm.pixel_array.astype(np.float32)
            img = (img - img.min()) / (img.max() - img.min() + 1e-6)
            img = cv2.resize(img, (224, 224))
            img = np.stack([img, img, img], axis=0)
            images.append(img)
        
        while len(images) < self.cfg.MAX_SLICES:
            images.append(np.zeros_like(images[0]))
        
        return torch.from_numpy(np.stack(images[:self.cfg.MAX_SLICES]))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, idx):
        study = self.studies[idx]
        study_id = study['study_id']
        
        all_images, all_plane_ids, all_fluid, all_fat = [], [], [], []
        for series_info in study['series']:
            images = self._load_series(series_info['series_id'])
            all_images.append(images)
            all_plane_ids.append(series_info['plane_id'])
            all_fluid.append(series_info['fluid_sensitive'])
            all_fat.append(series_info['fat_suppression'])
        
        labels = torch.tensor(self.labels_df.loc[study_id, cfg.LABELS].values, dtype=torch.float32)
        
        return {
            'images_list': all_images,
            'plane_ids': torch.tensor(all_plane_ids, dtype=torch.long),
            'fluid_sensitive': torch.tensor(all_fluid, dtype=torch.long),
            'fat_suppression': torch.tensor(all_fat, dtype=torch.long),
            'labels': labels,
        }

In [14]:
# ── Collate function ──────────────────────────────────────────────────────────
import cv2

def collate_fn(batch: List[Dict]) -> Dict:
    all_images, all_plane_ids, all_fluid, all_fat, all_labels = [], [], [], [], []
    max_series = max(len(item['images_list']) for item in batch)
    
    for item in batch:
        all_images.extend(item['images_list'])
        all_plane_ids.extend(item['plane_ids'])
        all_fluid.extend(item['fluid_sensitive'])
        all_fat.extend(item['fat_suppression'])
        all_labels.append(item['labels'])
        
        num_to_pad = max_series - len(item['images_list'])
        if num_to_pad > 0:
            dummy_img = torch.zeros_like(item['images_list'][0])
            all_images.extend([dummy_img] * num_to_pad)
            all_plane_ids.extend([0] * num_to_pad)
            all_fluid.extend([0] * num_to_pad)
            all_fat.extend([0] * num_to_pad)
    
    images = torch.stack(all_images)
    images = pad_image(images, patch_size=14)  # Pad to 14x multiples
    
    return {
        'images': images,
        'plane_ids': torch.tensor(all_plane_ids, dtype=torch.long),
        'fluid_sensitive': torch.tensor(all_fluid, dtype=torch.long),
        'fat_suppression': torch.tensor(all_fat, dtype=torch.long),
        'labels': torch.stack(all_labels),
        'num_series': max_series,
    }

In [15]:
# ── Loss and Training Utilities ────────────────────────────────────────────────
class WeightedBCELoss(nn.Module):
    '''Weighted BCE for imbalanced labels'''
    def __init__(self, pos_weight=None):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, pred, target):
        if self.pos_weight is not None:
            loss = F.binary_cross_entropy_with_logits(pred, target, pos_weight=self.pos_weight)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss

def compute_class_weights(df_labels, cfg):
    """Compute positive class weights"""
    weights = []
    for label in cfg.LABELS:
        pos = (df_labels[label] == 1).sum()
        neg = (df_labels[label] == 0).sum()
        weight = neg / (pos + 1e-6)
        weights.append(weight)
    return torch.tensor(weights, dtype=torch.float32)

print('✓ Loss and training utilities ready')

✓ Loss and training utilities ready


In [16]:
def train_epoch(model, loader, optimizer, criterion, cfg, scaler=None):
    model.train()
    losses = []
    
    pbar = tqdm(loader, desc='Train')
    for batch in pbar:
        images = batch['images'].to(cfg.DEVICE)  # (B*num_series, S, 3, H, W)
        labels = batch['labels'].to(cfg.DEVICE)  # (B, 12)
        plane_ids = batch['plane_ids'].to(cfg.DEVICE)
        fluid_sensitive = batch['fluid_sensitive'].to(cfg.DEVICE)
        fat_suppression = batch['fat_suppression'].to(cfg.DEVICE)
        
        # Mark zero-padded series (last num_series-1 per sample, if padded)
        slice_mask = torch.zeros(images.shape[0], images.shape[1], dtype=torch.bool, device=cfg.DEVICE)
        
        optimizer.zero_grad()
        
        if cfg.USE_AMP and scaler:
            with torch.autocast(device_type='cuda' if cfg.DEVICE.type == 'cuda' else 'cpu'):
                logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{np.mean(losses):.4f}'})
    
    return np.mean(losses)

In [17]:
@torch.no_grad()
def eval_epoch(model, loader, criterion, cfg):
    model.eval()
    losses = []
    all_preds, all_labels = [], []
    
    for batch in tqdm(loader, desc='Val'):
        images = batch['images'].to(cfg.DEVICE)
        labels = batch['labels'].to(cfg.DEVICE)
        plane_ids = batch['plane_ids'].to(cfg.DEVICE)
        fluid_sensitive = batch['fluid_sensitive'].to(cfg.DEVICE)
        fat_suppression = batch['fat_suppression'].to(cfg.DEVICE)
        
        slice_mask = torch.zeros(images.shape[0], images.shape[1], dtype=torch.bool, device=cfg.DEVICE)
        
        logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    val_loss = np.mean(losses)
    val_auc = roc_auc_score(all_labels, all_preds, average='macro')
    
    return val_loss, val_auc

In [18]:
def main():
    print('Loading data...')
    df_labels = pd.read_csv(cfg.LABELS_CSV)
    pos_weights = compute_class_weights(df_labels, cfg).to(cfg.DEVICE)
    print(f'Pos weights (first 3): {pos_weights[:3].cpu().numpy()}')
    
    ds_train = KneeTrainDataset(df_labels, cfg.TRAIN_IMG_DIR, cfg, split='train')
    ds_val = KneeTrainDataset(df_labels, cfg.TRAIN_IMG_DIR, cfg, split='val')
    print(f'Train: {len(ds_train)}, Val: {len(ds_val)}')
    
    ld_train = DataLoader(ds_train, batch_size=cfg.BATCH_SIZE, shuffle=True,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn, drop_last=True)
    ld_val = DataLoader(ds_val, batch_size=cfg.BATCH_SIZE, shuffle=False,
                       num_workers=cfg.NUM_WORKERS, pin_memory=True,
                       collate_fn=collate_fn)
    
    # Initialize model
    print('\\nInitializing model...')
    model = KneeAbnormalityModelWithMedDINOv3(cfg).to(cfg.DEVICE)
    
    # Verify MedDINOv3 backbone is frozen
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable_params:,} / Total: {total_params:,}  ({100*trainable_params/total_params:.1f}%)')
    print('MedDINOv3 backbone frozen ✓')
    
    # Setup optimizer and loss
    trainable_params_list = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params_list, lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    criterion = WeightedBCELoss(pos_weight=pos_weights)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler() if cfg.USE_AMP else None
    
    best_auc = 0
    for epoch in range(cfg.NUM_EPOCHS):
        print(f'\\n--- Epoch {epoch+1}/{cfg.NUM_EPOCHS} ---')
        train_loss = train_epoch(model, ld_train, optimizer, criterion, cfg, scaler)
        val_loss, val_auc = eval_epoch(model, ld_val, criterion, cfg)
        scheduler.step()
        
        print(f'Train loss: {train_loss:.4f}  Val loss: {val_loss:.4f}  Val AUC: {val_auc:.4f}')
        
        if val_auc > best_auc:
            best_auc = val_auc
            ckpt = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auc': val_auc,
            }
            torch.save(ckpt, cfg.CHECKPOINT_DIR / 'best_model_meddinov3.pth')
            print(f'✓ Checkpoint saved (AUC: {val_auc:.4f})')
    
    print(f'\\n✓ Training complete. Best AUC: {best_auc:.4f}')

if __name__ == '__main__':
    main()

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection/train_series_descriptions.csv'

In [ ]:
# To run training:
# main()